# 03 — Evaluate & Predict
**Section A:** Run validation metrics on test set  
**Section B:** Run inference on sample images

---
## Section A — Evaluation

In [ ]:
# Cell A1 — Load best.pt
from ultralytics import YOLO

MODEL_PATH = "../models/best.pt"
DATA_YAML  = "../flir-data-set-27/data.yaml"   # same yaml used during training

model = YOLO(MODEL_PATH)
print(f"Loaded: {MODEL_PATH}")

In [ ]:
# Cell A2 — Run model.val() on the test split
# YOLOv8 saves plots (confusion matrix, PR curve, etc.) inside runs/val/
metrics = model.val(
    data   = DATA_YAML,
    split  = "test",          # evaluate on test set
    imgsz  = 640,
    conf   = 0.3,
    iou    = 0.5,
    plots  = True,            # save confusion_matrix.png, PR_curve.png etc.
)

VAL_DIR = str(metrics.save_dir)   # keep for displaying plots below
print(f"\nVal results saved to: {VAL_DIR}")

In [ ]:
# Cell A3 — Print evaluation metrics
print("=" * 40)
print("  Evaluation Metrics (Test Set)")
print("=" * 40)
print(f"  mAP@0.5      : {metrics.box.map50:.4f}")
print(f"  mAP@0.5:0.95 : {metrics.box.map:.4f}")
print(f"  Precision     : {metrics.box.mp:.4f}")
print(f"  Recall        : {metrics.box.mr:.4f}")
print("=" * 40)

In [ ]:
# Cell A4 — Display confusion_matrix.png
import os
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

cm_path = os.path.join(VAL_DIR, "confusion_matrix.png")

img = mpimg.imread(cm_path)
plt.figure(figsize=(8, 6))
plt.imshow(img)
plt.axis("off")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
# Cell A5 — Display PR_curve.png
pr_path = os.path.join(VAL_DIR, "PR_curve.png")

img = mpimg.imread(pr_path)
plt.figure(figsize=(8, 6))
plt.imshow(img)
plt.axis("off")
plt.title("Precision-Recall Curve")
plt.tight_layout()
plt.show()

---
## Section B — Prediction

In [ ]:
# Cell B1 — Pick 6 random images from test/images
import random
import cv2
import numpy as np

TEST_IMG_DIR = "../flir-data-set-27/test/images"   # adjust if your path differs
CLASS_NAMES  = ["person", "car", "bicycle", "dog"]

img_exts = (".jpg", ".jpeg", ".png", ".bmp")
all_imgs = [os.path.join(TEST_IMG_DIR, f)
            for f in os.listdir(TEST_IMG_DIR)
            if f.lower().endswith(img_exts)]

random.seed(7)
sample_paths = random.sample(all_imgs, min(6, len(all_imgs)))
print(f"Selected {len(sample_paths)} images from: {TEST_IMG_DIR}")

In [ ]:
# Cell B2 — Run model.predict() on the 6 images
preds = model.predict(
    source = sample_paths,
    conf   = 0.3,
    iou    = 0.45,
    imgsz  = 640,
    verbose= False,
)

print(f"Inference done on {len(preds)} images")

In [ ]:
# Cell B3 — Display results in a 2x3 grid with bounding boxes and labels
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle("Model Predictions on Test Images (conf ≥ 0.3)", fontsize=13)

for ax, result in zip(axes.flat, preds):
    # result.plot() returns a BGR numpy array with boxes drawn
    annotated = result.plot()           # BGR
    annotated = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)

    ax.imshow(annotated)
    ax.set_title(os.path.basename(result.path)[:28], fontsize=7)
    ax.axis("off")

plt.tight_layout()
plt.savefig("../data/predictions_sample.png", dpi=150)
plt.show()
print("Saved: data/predictions_sample.png")

In [ ]:
# Cell B4 — Detection summary: count of each class across all 6 images
from collections import Counter

total_counts = Counter()

for result in preds:
    for cls_id in result.boxes.cls.cpu().numpy().astype(int):
        total_counts[CLASS_NAMES[cls_id]] += 1

print("Detection summary across 6 test images:")
print("-" * 30)
for cls in CLASS_NAMES:
    print(f"  {cls:<10}: {total_counts.get(cls, 0):>4} detections")
print("-" * 30)
print(f"  {'TOTAL':<10}: {sum(total_counts.values()):>4} detections")